In [ ]:
# @title 1.1 🔍 Check GPU Environment
import torch

print("="*50)
print("🔍 GPU Environment Check")
print("="*50)

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    total_mem = 0
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1024**3
        total_mem += mem_gb
        print(f"   GPU {i}: {props.name}")
        print(f"          Memory: {mem_gb:.1f} GB")
        print(f"          Compute: {props.major}.{props.minor}")
    print(f"\n   📊 Total VRAM: {total_mem:.1f} GB")
    
    # Recommendation
    if total_mem >= 30:
        print("   ✅ Excellent! Can run all models at full precision.")
    elif total_mem >= 15:
        print("   ✅ Good! Using 4-bit quantization for efficiency.")
    else:
        print("   ⚠️ Limited VRAM. Some features may be restricted.")
else:
    print("❌ No GPU detected!")
    print("   Enable GPU: Runtime → Change runtime type → GPU")

print("="*50)

In [ ]:
# @title 1.2 📦 Clone Repository & Install Dependencies
import os
import sys

# Configuration
REPO_URL = "https://github.com/ngnam1104/TriMedAgent.git"
WORK_DIR = "/kaggle/working/TriMedAgent"

print("📥 Setting up TriMedAgent...")

# Clone
if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
    print("   ✅ Repository cloned")
else:
    print("   ✅ Repository exists")

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# Install core dependencies
print("\n📦 Installing dependencies...")
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q open_clip_torch einops timm safetensors
!pip install -q sentencepiece pillow requests scipy
!pip install -q gradio nest_asyncio protobuf

# Install detection/segmentation tools
print("   + Grounding DINO...")
!pip install -q groundingdino-py

print("   + Segment Anything...")
!pip install -q segment-anything

print("\n✅ All dependencies installed!")
print("⚠️ If import errors occur, restart the kernel and run from cell 1.3")

In [ ]:
# @title 1.3 📥 Download Model Weights
import os
import requests
from tqdm import tqdm
from pathlib import Path

def download_file(url, filepath, desc=None):
    """Download file with progress bar"""
    filepath = Path(filepath)
    if filepath.exists():
        print(f"   ✅ {filepath.name} (cached)")
        return
    
    filepath.parent.mkdir(parents=True, exist_ok=True)
    desc = desc or filepath.name
    
    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))
    
    with open(filepath, 'wb') as f:
        with tqdm(total=total, unit='iB', unit_scale=True, desc=f"   📥 {desc}") as bar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                bar.update(len(chunk))

print("📥 Downloading model weights...")
print()

# Grounding DINO
download_file(
    "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth",
    "weights/groundingdino_swint_ogc.pth",
    "GroundingDINO"
)
download_file(
    "https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py",
    "weights/GroundingDINO_SwinT_OGC.py",
    "DINO Config"
)

# MedSAM / SAM
download_file(
    "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
    "weights/medsam_vit_b.pth",
    "MedSAM"
)

print("\n✅ All weights ready!")

---
## 2️⃣ 🔧 Load AI Models

In [ ]:
# @title 2.1 📚 Import TriMedAgent Modules
import sys
from pathlib import Path

# Add to path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import modules
from src import (
    TriMedOrchestrator,
    BiomedCLIPTool,
    GroundingDINO,
    MedSAMWrapper,
    LLaVABrain,
)

from src.utils.visualization import (
    draw_boxes_on_image,
    draw_masks_on_image,
)

print("✅ Imports successful!")

In [ ]:
# @title 2.2 ⚙️ Configuration
import torch

# Auto-detect GPU configuration
n_gpus = torch.cuda.device_count()
USE_4BIT = True  # Always use 4-bit for efficiency

if n_gpus >= 2:
    # Multi-GPU: distribute models
    DEVICE_MAP = {
        "llava": "cuda:0",
        "biomedclip": "cuda:1",
        "grounding_dino": "cuda:1",
        "medsam": "cuda:1"
    }
    print("🎮 Multi-GPU mode: Models distributed across 2 GPUs")
else:
    # Single GPU: all on cuda:0
    DEVICE_MAP = {
        "llava": "cuda:0",
        "biomedclip": "cuda:0",
        "grounding_dino": "cuda:0",
        "medsam": "cuda:0"
    }
    print("🎮 Single-GPU mode: All models on cuda:0")

print(f"   4-bit Quantization: {'Enabled' if USE_4BIT else 'Disabled'}")

In [ ]:
# @title 2.3 🧠 Load LLaVA-Med (Brain)
import gc

print(f"🧠 Loading LLaVA-Med on {DEVICE_MAP['llava']}...")
print("   This may take 2-3 minutes...")

llava_tool = LLaVABrain(
    device=DEVICE_MAP['llava'],
    quantize_4bit=USE_4BIT,
    load_on_init=True
)

# Clear cache
gc.collect()
torch.cuda.empty_cache()

print("✅ LLaVA-Med ready!")

In [ ]:
# @title 2.4 🔬 Load BiomedCLIP (Triage)
print(f"🔬 Loading BiomedCLIP on {DEVICE_MAP['biomedclip']}...")

biomedclip_tool = BiomedCLIPTool(
    device=DEVICE_MAP['biomedclip'],
    load_on_init=True
)

print("✅ BiomedCLIP ready!")

In [ ]:
# @title 2.5 🎯 Load Grounding DINO (Detection)
print(f"🎯 Loading Grounding DINO on {DEVICE_MAP['grounding_dino']}...")

dino_tool = GroundingDINO(
    config_path="weights/GroundingDINO_SwinT_OGC.py",
    checkpoint_path="weights/groundingdino_swint_ogc.pth",
    device=DEVICE_MAP['grounding_dino'],
    box_threshold=0.25
)
dino_tool.load_model()

print("✅ Grounding DINO ready!")

In [ ]:
# @title 2.6 🎭 Load MedSAM (Segmentation)
print(f"🎭 Loading MedSAM on {DEVICE_MAP['medsam']}...")

medsam_tool = MedSAMWrapper(
    checkpoint_path="weights/medsam_vit_b.pth",
    device=DEVICE_MAP['medsam'],
    model_type="vit_b"
)
medsam_tool.load_model()

gc.collect()
torch.cuda.empty_cache()

print("✅ MedSAM ready!")
print("\n" + "="*50)
print("🎉 All models loaded successfully!")
print("="*50)

---
## 3️⃣ 🧪 Quick Test (Optional)

In [ ]:
# @title 3.1 📷 Load Sample Image
from PIL import Image
import matplotlib.pyplot as plt

# Try to load sample image
SAMPLE_PATH = "images/example_chest.jpg"

try:
    sample_image = Image.open(SAMPLE_PATH).convert("RGB")
    print(f"✅ Loaded: {SAMPLE_PATH}")
    print(f"   Size: {sample_image.size}")
except:
    # Create dummy image for testing
    import numpy as np
    sample_image = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))
    print("⚠️ Using placeholder image")

plt.figure(figsize=(6, 6))
plt.imshow(sample_image)
plt.title("Sample Medical Image")
plt.axis('off')
plt.show()

In [ ]:
# @title 3.2 🧪 Quick Model Tests
print("🧪 Running quick tests...\n")

# Test BiomedCLIP
print("1️⃣ BiomedCLIP Triage:")
triage_result = biomedclip_tool.classify(sample_image)
print(f"   Modality: {triage_result.get('modality', 'Unknown')}")
print(f"   Confidence: {triage_result.get('confidence', 0):.1%}")

# Test Grounding DINO
print("\n2️⃣ Grounding DINO Detection:")
detections = dino_tool.detect(sample_image, "abnormality")
print(f"   Found: {len(detections)} regions")

# Test LLaVA
print("\n3️⃣ LLaVA-Med Analysis:")
response = llava_tool.query(sample_image, "Describe this medical image briefly.")
print(f"   Response: {response[:200]}...")

print("\n✅ All tests passed!")

---
## 4️⃣ 🚀 Initialize Orchestrator

In [ ]:
# @title 4.1 🎯 Create Hybrid ReAct Orchestrator

# Configuration
config = {
    "device": {
        "llava_device": DEVICE_MAP['llava'],
        "tool_device": DEVICE_MAP['grounding_dino']
    },
    "models": {
        "llava": {
            "model_name": "chaoyinshe/llava-med-v1.5-mistral-7b-hf",
            "quantize_4bit": USE_4BIT
        },
        "grounding_dino": {
            "config_path": "weights/GroundingDINO_SwinT_OGC.py",
            "weights_path": "weights/groundingdino_swint_ogc.pth"
        },
        "medsam": {
            "checkpoint": "weights/medsam_vit_b.pth"
        },
        "biomedclip": {
            "model_name": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
        }
    },
    "pipeline": {
        "max_iterations": 5,
        "enable_verification": True
    }
}

# Create orchestrator
orchestrator = TriMedOrchestrator(config)

print("✅ Hybrid ReAct Orchestrator initialized!")
print(f"   Max iterations: {config['pipeline']['max_iterations']}")
print(f"   Verification: {'Enabled' if config['pipeline']['enable_verification'] else 'Disabled'}")

In [ ]:
# @title 4.2 🔬 Test Full Pipeline
import time

print("🔬 Running Full Pipeline Test...")
print("="*50)

query = "Find any nodules or suspicious lesions in the lungs"
print(f"Query: '{query}'")
print()

start_time = time.time()
result = orchestrator.process(sample_image, query)
elapsed = time.time() - start_time

print(f"\n📊 Pipeline Result:")
print(f"   ✓ Success: {result.success}")
print(f"   ⏱️ Time: {elapsed:.1f}s")
print(f"   📍 Steps: {' → '.join(result.steps_executed)}")
print(f"   🔄 Iterations: {result.agent_iterations}")
print(f"   📋 Modality: {result.triage_modality} ({result.triage_confidence:.0%})")
print(f"   🎯 Detections: {len(result.verified_boxes)}")
print(f"   🎭 Masks: {len(result.masks)}")

In [ ]:
# @title 4.3 📊 Visualize Results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Original
axes[0].imshow(sample_image)
axes[0].set_title("Original Image", fontsize=14)
axes[0].axis('off')

# Result
if result.annotated_image:
    axes[1].imshow(result.annotated_image)
    axes[1].set_title(f"Analysis Result ({len(result.verified_boxes)} detections)", fontsize=14)
else:
    axes[1].imshow(sample_image)
    axes[1].set_title("No detections", fontsize=14)
axes[1].axis('off')

plt.suptitle(f"Query: '{query}'", fontsize=12, style='italic')
plt.tight_layout()
plt.show()

# Show report
if result.final_report:
    print("\n📝 Analysis Report:")
    print("-"*50)
    print(result.final_report)

---
## 5️⃣ 🎨 Launch Gradio Web Interface

In [ ]:
# @title 5.1 🚀 Launch Interactive Demo
from src.ui import launch_demo

print("🎨 Launching TriMedAgent Web Interface...")
print()
print("Features:")
print("   📷 Upload any medical image (X-ray, CT, MRI)")
print("   💬 Ask questions in natural language")
print("   🎯 Automatic detection & segmentation")
print("   🧠 AI-powered analysis reports")
print()
print("="*50)

# Launch with public URL for Kaggle
launch_demo(
    orchestrator=orchestrator,
    share=True,  # Creates public URL
    port=7860,
    debug=False
)

---
## 📚 Usage Examples

### Example Queries

**Chest X-ray:**
- "Find any nodules in the lungs"
- "Check for cardiomegaly"
- "Is there pleural effusion?"
- "Look for signs of pneumonia"

**CT Scan:**
- "Detect any tumors"
- "Find abnormal masses"
- "Measure the lesion size"

**MRI:**
- "Identify any brain lesions"
- "Check for abnormalities in the tissue"

---

## ⚠️ Disclaimer

**This tool is for research and educational purposes only.**

- Not intended for clinical diagnosis
- Always consult qualified healthcare professionals
- Results should be verified by medical experts

---

## 🔗 Links

- **GitHub**: [TriMedAgent](https://github.com/ngnam1104/TriMedAgent)
- **Models**: [HuggingFace](https://huggingface.co/ngnam1104)
- **Paper**: Coming soon

In [ ]:
# @title 6.1 💾 (Optional) Save Session State
# Run this before closing to save your session

import pickle
from pathlib import Path

session_dir = Path("/kaggle/working/session")
session_dir.mkdir(exist_ok=True)

# Save configuration
with open(session_dir / "config.pkl", 'wb') as f:
    pickle.dump({
        'device_map': DEVICE_MAP,
        'use_4bit': USE_4BIT,
        'config': config
    }, f)

print(f"✅ Session saved to {session_dir}")
print("\n🎉 Thank you for using TriMedAgent!")